In [11]:
from BCP303.BCP303 import BPC303
from Sourcemeter.sourcemeter import Sourcemeter2401
import time
from datetime import datetime
from tool.tools import post_process


In [ ]:
# move the stage in Z direction
bcp303_z = BPC303(channel_id=1)
position_z = bcp303_z.move_to_origin(start_position=0)
step_size_z = 0.1  # move 0.1 mm
position_z = bcp303_z.bcp303_move_stage(step_size=step_size_z, current_position=position_z,)
bcp303_z.bcp303_stop(ifback=True)

In [12]:
# move the stage in x direction
bcp303_x = BPC303(channel_id=2)
position_x = bcp303_x.move_to_origin(start_position=0)
step_size_x = 0.1  # move 0.1 mm
position_x = bcp303_x.bcp303_move_stage(step_size=step_size_x, current_position=position_x,)
bcp303_x.bcp303_stop(ifback=False)

APT High Voltage Amplifier initialized
Stage Moving Done
Stage disconnected


In [ ]:
# record the signals
sm2401 = Sourcemeter2401(speed_nplc=0.1)


In [10]:
# sweep operation: move stage in z-direction and record the signals

def sweep_operation(stage_settings=None,chip_name="chip_test",sample_name="beam_test"):
    step_number = stage_settings["step_number"]
    time_interval = stage_settings["time_interval"]
    position_x = stage_settings["position_x"]
    step_size_z = stage_settings["step_size_z"]
    count = 1
    formatted_time = datetime.now().strftime("%Y%m%d%H%M")
    try:
        bcp303_z = BPC303(channel_id=1)
        bcp303_device = bcp303_z.get_device()
        # moving controller x
        bcp303_x = BPC303(channel_id=2, device=bcp303_device)
        # Sourcemeter
        sm2401 = Sourcemeter2401(speed_nplc=0.1)
        # set the initial position of the stage z
        position_z = bcp303_z.move_to_origin(start_position=stage_settings["position_z"]) - step_size_z
        time.sleep(1)
        bcp303_x.move_to_origin()
        time.sleep(1)
        allData = [{"position": [], "voltage": []}]
        for _ in range(step_number):
            position_z = bcp303_z.bcp303_move_stage(
                step_size=step_size_z,
                current_position=position_z,
            )
            allData[0]["position"].append(position_z)
            time.sleep(0.5)
            bcp303_x.move_to_position(start_position=0, step_size=0.5, final_position=position_x)
            time.sleep(0.5)
            step_start = time.perf_counter()
            voltage = sm2401.measure_voltage(duration=time_interval / 2, dt=0.01)[
                "voltage"
            ]
            allData[0]["voltage"].append(voltage)
            # remaining time to wait until the next step
            elapsed = time.perf_counter() - step_start
            remaining = time_interval - elapsed
            if remaining > 0:
                time.sleep(remaining)
            bcp303_x.move_to_origin()
            time.sleep(0.5)
            print(f"process completed: {100 * count / step_number:.1f}%")
            count += 1
        sm2401_settings = sm2401.getSettings()
        stage_settings["position_x"] = position_x
        settings = {"stage": stage_settings, "sourcemeter": sm2401_settings}
        post_process(
            chip_name=chip_name,
            sample_name=sample_name,
            result=allData[0],
            config=settings,
            repeat=1,
            position_z=position_z,
            ifshow=False,
            formatted_time=formatted_time,
        )
    # stop the stage and sourcemeter
    except Exception as e:
        print(f"Exception occurred: {e}")
    finally:
        time.sleep(0.5)
        sm2401.close()
        bcp303_z.move_to_origin()
        bcp303_z.channel.StopPolling()
        bcp303_x.bcp303_stop(ifback=True)
        return allData[0], settings
    

setting = {
        "position_x": 1.5,
        "step_number": 20,
        "position_z": 0,
        "step_size_z": 1,
        "time_interval": 2,  
    }
sweep_operation(
        stage_settings=setting,
        chip_name="V1_R_W_1_Left",
        # chip_name="SiN_beam",
        # sample_name="w2_sweep",  # test_1_right, w=20
        sample_name="test_AFM4_450_w20_4",  
    )

APT High Voltage Amplifier initialized
APT High Voltage Amplifier initialized
IDN: KEITHLEY INSTRUMENTS INC.,MODEL 2401,4614233,B02 Jan 20 2021 10:19:49/B01  /W/N
process completed: 5.0%
process completed: 10.0%
process completed: 15.0%
process completed: 20.0%
process completed: 25.0%
process completed: 30.0%
process completed: 35.0%
process completed: 40.0%
process completed: 45.0%
process completed: 50.0%
process completed: 55.0%
process completed: 60.0%
process completed: 65.0%
process completed: 70.0%
process completed: 75.0%
process completed: 80.0%
process completed: 85.0%
process completed: 90.0%
process completed: 95.0%
process completed: 100.0%
Config saved to: ./result/V1_R_W_1_Left/202606171343_test_AFM4_450_w20_4\202606171343_V1_R_W_1_Left_test_AFM4_450_w20_4_config.json
Stage Moving Done
Stage disconnected


({'position': [0.0,
   0.986358226264229,
   1.97149571214942,
   2.95419171727653,
   3.9509262367626,
   4.94644001586962,
   5.94012268440809,
   6.93929868465224,
   7.93115024262215,
   8.92300180059206,
   9.91180150761437,
   10.8975493636891,
   11.8961149937437,
   12.8769798883023,
   13.8541825617237,
   14.8271126438185,
   15.7951597643971,
   16.7955565050203,
   17.795342875454,
   18.7859736930448],
  'voltage': [[0.04560694,
    0.04585324,
    0.04581584,
    0.0457228,
    0.04632718,
    0.04600317,
    0.04607008,
    0.04595574,
    0.04601913,
    0.04627074,
    0.04593149,
    0.04603085,
    0.04631145,
    0.04605048,
    0.04549738,
    0.0458717],
   [0.04578966,
    0.04536601,
    0.04567405,
    0.04591012,
    0.04614228,
    0.0461072,
    0.04575874,
    0.0457512,
    0.04594881,
    0.0456961,
    0.04565554,
    0.0453629,
    0.04542211,
    0.04543791,
    0.04519106,
    0.04527908],
   [0.04526328,
    0.04529028,
    0.04489884,
    0.0448057,